# Indexing Chunks Using ChromaDB

**Indexing** is the process of storing your embedded chunks into a vector database so they can be searched later. ChromaDB is a lightweight, local vector database that stores both the vectors and the original text together.

---

## How Indexing Works

```
Text Chunks  →  Embedding Model  →  Vectors  →  ChromaDB (stored on disk)
"AI is..."       [0.21, -0.45...]              { id, vector, text, metadata }
```

Each chunk is stored as a **document** in ChromaDB with:
- **ID** — unique identifier for the chunk
- **Embedding** — the vector representation
- **Document** — the original chunk text
- **Metadata** — optional info like source file, page number, chunk index

---

## Setup

```bash
pip install chromadb sentence-transformers
```

---

## Full Indexing Example

```python
import chromadb
from sentence_transformers import SentenceTransformer

# 1. Your chunks (from splitting a document)
chunks = [
    "Machine learning is a subset of AI.",
    "AI enables computers to learn from data.",
    "Learning improves with more training data.",
    "Neural networks are inspired by the human brain.",
]

# 2. Load local embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Create embeddings for all chunks
embeddings = model.encode(chunks).tolist()

# 4. Setup ChromaDB (persists to disk)
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="my_docs")

# 5. Index the chunks into ChromaDB
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=embeddings,
    documents=chunks,
    metadatas=[{"chunk_index": i} for i in range(len(chunks))]
)

print(f"Indexed {collection.count()} chunks")
```

---

## Querying the Index

```python
# Embed the user query using the same model
query = "How do machines learn?"
query_embedding = model.encode([query]).tolist()

# Search ChromaDB for top 2 most relevant chunks
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)

for doc, score in zip(results["documents"][0], results["distances"][0]):
    print(f"Score: {score:.4f} | Chunk: {doc}")
```

**Output:**
```
Score: 0.2134 | Chunk: AI enables computers to learn from data.
Score: 0.3421 | Chunk: Machine learning is a subset of AI.
```

> **Note:** ChromaDB uses distance scores — **lower = more similar**.

---

## With Metadata Filtering

Metadata lets you filter results by source, date, or any custom field:

```python
# Index with richer metadata
collection.add(
    ids=["chunk_0", "chunk_1"],
    embeddings=embeddings[:2],
    documents=chunks[:2],
    metadatas=[
        {"source": "ml_textbook.pdf", "page": 1},
        {"source": "ai_intro.pdf",    "page": 5},
    ]
)

# Query only chunks from a specific source
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2,
    where={"source": "ml_textbook.pdf"}   # filter by metadata
)
```

---

## ChromaDB Cheat Sheet

| Operation | Code |
|---|---|
| Create collection | `client.get_or_create_collection("name")` |
| Add chunks | `collection.add(ids, embeddings, documents, metadatas)` |
| Query top-K | `collection.query(query_embeddings, n_results=K)` |
| Count chunks | `collection.count()` |
| Delete a chunk | `collection.delete(ids=["chunk_0"])` |
| Peek at data | `collection.peek()` |

---

## Where ChromaDB Fits in RAG

```
Document → Chunking → Embedding → ChromaDB (index)
                                        ↓
                          Query → Embed Query → Search ChromaDB
                                        ↓
                               Top-K Chunks → LLM → Answer
```

> The index is built **once** and reused for every query — you don't re-embed the whole document each time.